# 10장. 선형 회귀의 계수 추정과 규제

`lstsq`는 계수를 어떻게 구하는가. 특성이 많을 때 계수를 왜, 얼마나 제한하는가.

**준비:** 2장에서 쓴 국토교통부 실거래가 공개시스템의 서울 아파트 매매 CSV 네 개와 6장에서 쓴 경기도 「일반음식점(중국식) 현황」 CSV를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. 끝까지 실행하는 데 3분쯤 걸린다.

## 10.1 최소제곱과 경사하강법: 특성의 단위와 학습률

In [ ]:
import os, re, glob
import numpy as np
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.width", 200)

def period(path):
    """파일 머리말에서 계약일자 범위를 읽는다. 실거래가 파일이 아니면 빈 리스트를 돌려준다."""
    with open(path, encoding="cp949", errors="ignore") as f:
        head = "".join(f.readline() for _ in range(15))
    return re.findall(r"계약일자 : (\d{4}-\d{2}-\d{2}) ~ (\d{4}-\d{2}-\d{2})", head)

files = sorted(f for f in os.listdir(".") if f.endswith(".csv") and period(f))
raw = pd.concat([pd.read_csv(f, encoding="cp949", skiprows=15, thousands=",",
                             dtype={"번지": str, "본번": str, "부번": str, "동": str}) for f in files],
                ignore_index=True)

v = raw[(raw["해제사유발생일"] == "-") & (raw["거래유형"] != "직거래")].copy()
v = v.rename(columns={"전용면적(㎡)": "면적"})
v["억"] = v["거래금액(만원)"] / 1e4
v["날짜"] = pd.to_datetime(v["계약년월"].astype(str) + v["계약일"].astype(str).str.zfill(2), format="%Y%m%d")
v["구"] = v["시군구"].str.split().str[1]
v["동네"] = v["시군구"].str.split().str[2]
v["동이름"] = v["구"] + " " + v["동네"]
v["단지"] = v["시군구"] + " " + v["번지"] + " " + v["단지명"]
v["연"] = v["날짜"].dt.year
v["연식"] = v["연"] - v["건축년도"]
v = v.sort_values("날짜").reset_index(drop=True)
print(len(files), raw.shape, len(v))

In [ ]:
a23, a24, a25 = v[v["연"] == 2023], v[v["연"] == 2024], v[v["연"] == 2025]
both = v[v["연"] <= 2024]
print(len(a23), len(a24), len(both), len(a25))

def four(df):
    """상수 열, 면적의 로그, 층, 건축년도."""
    return np.column_stack([np.ones(len(df)), np.log(df["면적"]), df["층"], df["건축년도"]])

X4, y4 = four(both), np.log(both["억"].to_numpy())
n = len(y4)
by_lstsq = np.linalg.lstsq(X4, y4, rcond=None)[0]
print(by_lstsq.round(4).tolist())

In [ ]:
by_hand = np.linalg.solve(X4.T @ X4, X4.T @ y4)          # 정규방정식 (XᵀX)β = Xᵀy를 푼다
print(by_hand.round(4).tolist(), f"{np.abs(by_hand - by_lstsq).max():.1e}")

def loss(M, coef):
    """평균제곱오차(MSE)."""
    return float(((y4 - M @ coef) ** 2).mean())

floor = loss(X4, by_hand)
print(round(floor, 4), round(float(y4.var()), 4), round(1 - floor / float(y4.var()), 3))
print((X4.T @ (y4 - X4 @ by_hand) / n).round(8).tolist())

In [ ]:
def descend(M, rate, steps):
    """계수 0에서 시작해서 기울기의 반대 방향으로 rate만큼씩 steps번 이동한다."""
    coef, seen = np.zeros(M.shape[1]), []
    for _ in range(steps):
        slope = 2 * M.T @ (M @ coef - y4) / n         # 손실을 계수마다 미분한 값(기울기)
        coef = coef - rate * slope
        seen.append(loss(M, coef))
    return coef, seen

raw_tracks = {}
with np.errstate(all="ignore"):
    for rate in [1e-6, 2e-7]:
        coef, raw_tracks[rate] = descend(X4, rate, 1000)
        print(rate, [float(f"{raw_tracks[rate][i]:.4g}") for i in (0, 1, 2, 9, 999)])
print(coef.round(4).tolist())

In [ ]:
print(np.abs(X4).mean(axis=0).round(1).tolist())
steep = np.linalg.eigvalsh(X4.T @ X4 / n)                 # 방향별 곡률(고윳값), 작은 것부터
print(f"{steep[-1]:.4g} {steep[0]:.4g} {steep[-1] / steep[0]:.3g}")
print(f"{1 / steep[-1]:.3g}")                             # 발산하지 않는 학습률의 한계
print(f"{np.log(2) / (2 * 2e-7 * steep[0]):.3g}")         # 곡률이 가장 작은 방향의 오차가 절반이 되는 데 필요한 횟수

In [ ]:
mu, sd = X4[:, 1:].mean(axis=0), X4[:, 1:].std(axis=0)
Z4 = np.column_stack([np.ones(n), (X4[:, 1:] - mu) / sd])     # 열마다 평균을 빼고 표준편차로 나눈다
steep = np.linalg.eigvalsh(Z4.T @ Z4 / n)
print(steep.round(3).tolist(), round(float(steep[-1] / steep[0]), 2), round(float(1 / steep[-1]), 3))

tracks = {}
with np.errstate(all="ignore"):
    for rate in [0.01, 0.1, 0.5, 0.9]:
        coef, tracks[rate] = descend(Z4, rate, 200)
        print(rate, [float(f"{tracks[rate][i]:.4g}") for i in (0, 2, 9, 49, 199)])

coef, _ = descend(Z4, 0.5, 100)
back = np.r_[coef[0] - (coef[1:] / sd * mu).sum(), coef[1:] / sd]     # 표준화한 계수를 원래 단위로 되돌린다
print(back.round(4).tolist(), int(np.argmin(tracks[0.9])) + 1)

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba
from matplotlib.ticker import FuncFormatter

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)


def plain_ticks(axis):
    """로그 축의 눈금을 10의 거듭제곱 대신 보통 숫자로 적는다."""
    axis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:,.0f}" if val >= 1 else f"{val:.7f}".rstrip("0")))

In [ ]:
# 그림 10-1
fig, ax = plt.subplots(figsize=(7, 3.9))
steps = np.arange(1, 1001)
big = np.array(raw_tracks[1e-6]); ok = np.isfinite(big) & (big < 1e9)
ax.plot(steps[ok], big[ok], color=BRICK, lw=1.8, marker="o", ms=4)
ax.plot(steps, raw_tracks[2e-7], color=NAVY, lw=1.8)
ax.axhline(float(y4.var()), color=GRAY, lw=1, ls=(0, (4, 2)))
ax.axhline(floor, color=TEAL, lw=1.2, ls=(0, (4, 2)))
ax.text(4.6, 2e4, "학습률 0.000001: 한 번 이동할 때마다\n손실이 약 49배가 된다(발산)", fontsize=8, color=BRICK, va="center", linespacing=1.5)
ax.text(30, 1.3, "학습률 0.0000002: 10번째부터 0.41에서 줄지 않는다", fontsize=8, color=NAVY, va="center")
ax.annotate(f"회색 점선: 목표의 분산 {y4.var():.4f}", xy=(12, y4.var()), xytext=(12, 9), fontsize=8, color="#5b6570", ha="center", va="center", arrowprops=dict(arrowstyle="->", color="#5b6570", lw=0.8))
ax.text(1000, floor * 0.86, f"최소제곱의 손실 {floor:.4f}", fontsize=8, color=TEAL, ha="right", va="top")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(0.9, 1100); ax.set_ylim(0.05, 5e6)
plain_ticks(ax.xaxis); plain_ticks(ax.yaxis)
ax.set_xlabel("이동 횟수(로그 축)", fontsize=8)
ax.set_ylabel("손실: 평균제곱오차(로그 축)", fontsize=8)
frame(ax)
fig.tight_layout()
fig.savefig("fig_10_1.png", dpi=300)
plt.show()

In [ ]:
P2, t2 = X4[:, 1:3] - X4[:, 1:3].mean(axis=0), y4 - y4.mean()      # 면적의 로그와 층만, 평균을 뺀 값
Q2 = P2 / P2.std(axis=0)                                           # 같은 두 열을 표준화한 것

def walk(M, rate, steps):
    """특성 두 개짜리 문제에서 경사하강법이 지나간 계수를 모두 기록한다."""
    coef, path = np.zeros(2), [np.zeros(2)]
    for _ in range(steps):
        coef = coef - rate * 2 * M.T @ (M @ coef - t2) / n
        path.append(coef.copy())
    return np.array(path)

for name, M, rate in [("원래 단위", P2, 0.02), ("표준화", Q2, 0.4)]:
    curve = np.linalg.eigvalsh(M.T @ M / n)
    path = walk(M, rate, 300)
    best = np.linalg.lstsq(M, t2, rcond=None)[0]
    close = np.abs(path - best).max(axis=1) < 0.01 * np.abs(best).max()
    print(name, curve.round(3).tolist(), round(float(curve[1] / curve[0]), 1), best.round(4).tolist(),
          path[300].round(4).tolist(), int(np.argmax(close)) if close.any() else "300번 안에 도달하지 못함")

In [ ]:
# 그림 10-2
fig, axes = plt.subplots(2, 1, figsize=(7, 7.4))
show = [("원래 단위: 면적의 로그와 층을 그대로 쓴 경우 (학습률 0.02, 300번)", P2, 0.02, (-0.08, 1.3), (-0.012, 0.047), "면적 로그의 계수", "층의 계수"),
        ("표준화한 경우 (학습률 0.4, 300번)", Q2, 0.4, (-0.05, 0.48), (-0.02, 0.12), "면적 로그(표준화)의 계수", "층(표준화)의 계수")]
for ax, (title, M, rate, xr, yr, xl, yl) in zip(axes, show):
    best = np.linalg.lstsq(M, t2, rcond=None)[0]
    Hm = M.T @ M / n
    g1, g2 = np.meshgrid(np.linspace(*xr, 300), np.linspace(*yr, 300))
    d1, d2 = g1 - best[0], g2 - best[1]
    extra = Hm[0, 0] * d1 ** 2 + 2 * Hm[0, 1] * d1 * d2 + Hm[1, 1] * d2 ** 2      # 최솟값보다 얼마나 큰가
    ax.contourf(g1, g2, extra, levels=[0, 0.002, 0.008, 0.02, 0.04, 0.07, 0.11, 0.16, 1], cmap="Greys", alpha=0.13)
    rings = ax.contour(g1, g2, extra, levels=[0.002, 0.008, 0.02, 0.04, 0.07, 0.11, 0.16], colors=GRAY, linewidths=0.8)
    ax.clabel(rings, levels=[0.002, 0.008, 0.02, 0.04, 0.07], fmt="+%.3g", fontsize=6.2, inline=True, inline_spacing=2, colors="#4b5563")       # 최솟값보다 큰 정도
    path = walk(M, rate, 300)
    ax.plot(path[:, 0], path[:, 1], color=BRICK, lw=1.1, marker="o", ms=2.6)
    ax.scatter([0], [0], s=46, color=NAVY, zorder=5)
    ax.scatter([best[0]], [best[1]], s=90, marker="*", color=TEAL, zorder=5)
    ax.set_xlim(*xr); ax.set_ylim(*yr)
    ax.set_xlabel(xl, fontsize=8); ax.set_ylabel(yl, fontsize=8)
    frame(ax, title, grid="both")
axes[0].annotate("시작(계수 0)", (0, 0), (0.06, -0.008), fontsize=8, color=NAVY, bbox=BOX, arrowprops={"arrowstyle": "->", "color": NAVY, "lw": 0.8})
axes[0].annotate("최소제곱의 계수", (np.linalg.lstsq(P2, t2, rcond=None)[0]), (0.93, 0.034), fontsize=8, color=TEAL, bbox=BOX, arrowprops={"arrowstyle": "->", "color": TEAL, "lw": 0.8})
axes[0].text(0.2, 0.040, "층의 계수는 위아래로 오르내리며 줄어들고\n면적 로그의 계수는 조금씩만 이동한다", fontsize=8, color=BRICK, linespacing=1.5, bbox=BOX)
axes[1].annotate("시작(계수 0)", (0, 0), (-0.04, 0.03), fontsize=8, color=NAVY, bbox=BOX, arrowprops={"arrowstyle": "->", "color": NAVY, "lw": 0.8})
axes[1].annotate("최소제곱의 계수", (np.linalg.lstsq(Q2, t2, rcond=None)[0]), (0.33, 0.04), fontsize=8, color=TEAL, bbox=BOX, arrowprops={"arrowstyle": "->", "color": TEAL, "lw": 0.8})
fig.tight_layout(h_pad=1.6)
fig.savefig("fig_10_2.png", dpi=300)
plt.show()

In [ ]:
# 그림 10-3
fig, ax = plt.subplots(figsize=(7, 3.9))
look = {0.01: (GRAY, "-"), 0.1: (NAVY, "-"), 0.5: (TEAL, "-"), 0.9: (BRICK, "-")}
for rate, (col, style) in look.items():
    ax.plot(range(1, 41), tracks[rate][:40], color=col, ls=style, lw=1.8)
ax.axhline(floor, color="#5b6570", lw=1, ls=(0, (4, 2)))
ax.text(40, floor * 0.9, f"최소제곱의 손실 {floor:.4f}", fontsize=8, color="#5b6570", ha="right", va="top")
ax.text(24, 2.4, "학습률 0.01: 줄어들지만 느리다", fontsize=8, color="#5b6570")
ax.text(9.5, 0.36, "학습률 0.1", fontsize=8, color=NAVY)
ax.text(1.6, 0.125, "학습률 0.5: 3번 만에 도달", fontsize=8, color=TEAL)
ax.text(15.5, 30, f"학습률 0.9: {int(np.argmin(tracks[0.9])) + 1}번째 이후 커진다(발산)", fontsize=8, color=BRICK, ha="right")
ax.set_yscale("log"); ax.set_ylim(0.1, 100); ax.set_xlim(0.5, 40.5)
plain_ticks(ax.yaxis)
ax.set_xlabel("이동 횟수", fontsize=8)
ax.set_ylabel("손실: 평균제곱오차(로그 축)", fontsize=8)
frame(ax)
fig.tight_layout()
fig.savefig("fig_10_3.png", dpi=300)
plt.show()

In [ ]:
rng = np.random.default_rng(0)
sgd_tracks = {}
for rate in [0.05, 0.005]:
    coef, marks, trail = np.zeros(4), [], []
    for lap in range(5):                                  # 자료 전체를 다섯 번 쓴다(5 에폭)
        order = rng.permutation(n)
        for i in range(0, n, 256):
            few = order[i: i + 256]                       # 거래 256건(미니배치)만으로 기울기를 추정한다
            coef = coef - rate * 2 * Z4[few].T @ (Z4[few] @ coef - y4[few]) / len(few)
            if (i // 256) % 5 == 0:
                trail.append(loss(Z4, coef))
        marks.append(round(loss(Z4, coef), 4))
    sgd_tracks[rate] = trail
    print(rate, marks)
print(len(range(0, n, 256)))

In [ ]:
# 그림 10-4
fig, ax = plt.subplots(figsize=(7, 3.9))
per_lap = len(range(0, n, 256))
for rate, col in [(0.05, BRICK), (0.005, NAVY)]:
    xs_ = (np.arange(len(sgd_tracks[rate])) * 5 + 1) / per_lap
    ax.plot(xs_, np.array(sgd_tracks[rate]) - floor, color=col, lw=1.2)
ax.text(4.9, 0.7, "학습률 0.05: 빨리 가까워지지만 0.0001~0.001에서 오르내린다", fontsize=8, color=BRICK, ha="right")
ax.text(4.9, 0.12, "학습률 0.005: 처음에는 느리지만 0.00001 근처까지 간다", fontsize=8, color=NAVY, ha="right")
for lap in range(1, 5):
    ax.axvline(lap, color="#d5dae0", lw=0.8, zorder=0)
ax.set_yscale("log"); ax.set_ylim(1e-7, 6); ax.set_xlim(0, 5)
plain_ticks(ax.yaxis)
ax.set_xlabel("에폭(자료 전체를 쓴 횟수)", fontsize=8)
ax.set_ylabel("손실 - 최소제곱의 손실(로그 축)", fontsize=8)
frame(ax)
fig.tight_layout()
fig.savefig("fig_10_4.png", dpi=300)
plt.show()

## 10.2 특성이 4,589개일 때: 릿지 회귀

In [ ]:
from scipy import sparse

def design(df, dongs, complexes):
    """수치 특성 7개에 동과 단지의 가변수를 붙인다. 0이 아닌 값만 저장하는 희소 행렬로 만든다."""
    num = np.column_stack([np.log(df["면적"]), df["층"],
                           *((np.clip(df["연식"], 0, 59) // 10 == k).astype(float) for k in range(1, 6))])
    blocks = [sparse.csr_matrix(num)]
    for col, cats in [("동이름", dongs), ("단지", complexes)]:
        where = pd.Index(cats).get_indexer(df[col])          # 목록에 없는 이름은 -1이 된다
        known = where >= 0
        blocks.append(sparse.csr_matrix((np.ones(known.sum()), (np.where(known)[0], where[known])), shape=(len(df), len(cats))))
    return sparse.hstack(blocks).tocsr()

D23, C23 = sorted(a23["동이름"].unique()), sorted(a23["단지"].unique())
S, S24 = design(a23, D23, C23), design(a24, D23, C23)
y23, y24 = np.log(a23["억"].to_numpy()), np.log(a24["억"].to_numpy())
print(S.shape, len(D23), len(C23), S.nnz, round(S.nnz / (S.shape[0] * S.shape[1]) * 100, 2))
deals = a23["단지"].value_counts()
print(int((deals == 1).sum()), int((deals <= 3).sum()), int(deals.median()), round(float((~a24["단지"].isin(C23)).mean()) * 100, 1))

In [ ]:
from sklearn.linear_model import Ridge

def off(actual_log, predicted_log):
    """오차율의 중앙값(%). 2장의 off와 같은 지표를 로그 가격에서 계산한다."""
    return round(float(np.median(np.abs(np.exp(predicted_log - actual_log) - 1)) * 100), 1)

loose = Ridge(alpha=1e-6, solver="sparse_cg", tol=1e-8, max_iter=20000).fit(S, y23)
print(off(y23, loose.predict(S)), off(y24, loose.predict(S24)))

shift = np.zeros(S.shape[1])
d = D23.index("서초구 반포동")
shift[7 + d] = 1.0                                           # 반포동의 계수를 1 올리고
inside = [7 + len(D23) + j for j, c in enumerate(C23) if c.startswith("서울특별시 서초구 반포동 ")]
shift[inside] = -1.0                                         # 반포동에 있는 단지의 계수를 모두 1 내린다
print(len(inside), f"{np.abs(S @ (loose.coef_ + shift) - S @ loose.coef_).max():.1e}")
print(round(float((loose.coef_ ** 2).sum()), 1), round(float(((loose.coef_ + shift) ** 2).sum()), 1))

In [ ]:
def ridge(M, target, lam):
    """제곱 오차의 합 + lam × (계수 제곱의 합)을 최소로 하는 계수와 절편. 절편에는 규제를 걸지 않는다."""
    mid, top = M.mean(axis=0), target.mean()
    C_ = M - mid
    coef = np.linalg.solve(C_.T @ C_ + lam * np.eye(M.shape[1]), C_.T @ (target - top))
    return coef, top - mid @ coef

small = S[:, : 7 + len(D23)].toarray()                       # 수치 특성과 동까지만. 315열이면 일반 배열로도 풀 수 있다
mine, base = ridge(small, y23, 3.0)
theirs = Ridge(alpha=3.0).fit(small, y23)
print(small.shape, f"{np.abs(mine - theirs.coef_).max():.1e}", f"{abs(base - theirs.intercept_):.1e}")

In [ ]:
by_dong = Ridge(alpha=0.01).fit(S[:, : 7 + len(D23)], y23)          # 수치 특성과 동까지만 쓴 모델
left23 = y23 - by_dong.predict(S[:, : 7 + len(D23)])                # 그 모델의 잔차
P = S[:, 7 + len(D23):]                                             # 단지의 가변수만
count = deals.reindex(C23).to_numpy()
plain = np.asarray(P.T @ left23).ravel() / count                    # 단지별 잔차의 평균(규제가 없을 때의 계수)
tied = Ridge(alpha=10, fit_intercept=False, solver="sparse_cg", tol=1e-10).fit(P, left23).coef_

rows = []
for lo, hi in [(1, 1), (2, 3), (4, 9), (10, 29), (30, 99), (100, 999)]:
    g = (count >= lo) & (count <= hi)
    rows.append((f"{lo}~{hi}건", int(g.sum()), round(float(np.abs(plain[g]).mean()), 3), round(float(np.abs(tied[g]).mean()), 3),
                 round(float((tied[g] / plain[g]).mean()), 3), round(float((count[g] / (count[g] + 10)).mean()), 3)))
print(pd.DataFrame(rows, columns=["2023년 거래", "단지 수", "평균의 크기", "릿지 계수의 크기", "남은 비율", "n/(n+10)"]).set_index("2023년 거래").to_string())

In [ ]:
# 그림 10-6
fig, ax = plt.subplots(figsize=(7, 4.6))
groups = [((count >= 30), TEAL, "거래 30건 이상인 단지(237곳)"), ((count >= 4) & (count <= 9), NAVY, "거래 4~9건인 단지(978곳)"), ((count == 1), BRICK, "거래 1건인 단지(1,414곳)")]
for g, col, label in groups:
    ax.scatter(plain[g], tied[g], s=7, color=to_rgba(col, 0.45), edgecolor="none", label=label)
line = np.array([-1.1, 1.1])
ax.plot(line, line, color=GRAY, lw=1, ls=(0, (4, 2)))
ax.plot(line, line / 11, color=BRICK, lw=0.9, ls=(0, (1, 2)))
ax.text(0.5, 0.6, "줄어들지 않을 때\n(기울기 1)", fontsize=8, color="#5b6570", ha="right", va="center", linespacing=1.5)
ax.text(1.08, 0.165, "기울기 1/11", fontsize=8, color=BRICK, ha="right")
ax.axhline(0, color="#c9d1da", lw=0.8, zorder=0); ax.axvline(0, color="#c9d1da", lw=0.8, zorder=0)
ax.set_xlim(-1.1, 1.1); ax.set_ylim(-0.5, 0.72)
ax.set_xlabel("규제가 없을 때의 단지 계수(단지별 잔차의 평균)", fontsize=8)
ax.set_ylabel("릿지(λ = 10)의 단지 계수", fontsize=8)
leg = ax.legend(frameon=False, fontsize=8, loc="upper left", markerscale=2.4)
frame(ax, grid="both")
fig.tight_layout()
fig.savefig("fig_10_6.png", dpi=300)
plt.show()

In [ ]:
LAMS = [0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100]
fits = {lam: Ridge(alpha=lam, solver="sparse_cg", tol=1e-8, max_iter=20000).fit(S, y23) for lam in LAMS}
rows = [(lam, off(y23, m.predict(S)), off(y24, m.predict(S24)), round(float(np.sqrt(((m.predict(S24) - y24) ** 2).mean())), 4),
         round(float(np.abs(m.coef_[7 + len(D23):]).mean()), 3)) for lam, m in fits.items()]
apt = pd.DataFrame(rows, columns=["λ", "2023년 %", "2024년 %", "2024년 로그 RMSE", "단지 계수의 크기"]).set_index("λ")
print(apt.to_string())

In [ ]:
# 그림 10-7
fig, ax = plt.subplots(figsize=(7, 3.9))
ax.plot(apt.index, apt["2024년 %"], color=BRICK, lw=1.8, marker="o", ms=5, label="2024년(검증 자료)")
ax.plot(apt.index, apt["2023년 %"], color=NAVY, lw=1.8, marker="s", ms=5, label="2023년(훈련 자료)")
ax.legend(frameon=False, fontsize=8, loc="upper left")
ax.annotate(f"λ = 0.01에서 {apt['2024년 %'].iloc[0]}%로 가장 낮다", (0.01, apt["2024년 %"].iloc[0]), (0.03, 10.6), fontsize=8, color=BRICK, arrowprops={"arrowstyle": "->", "color": BRICK, "lw": 0.8})
ax.set_xscale("log"); ax.set_ylim(0, 19)
plain_ticks(ax.xaxis)
ax.set_xlabel("규제 세기 λ(로그 축)", fontsize=8)
ax.set_ylabel("오차율의 중앙값(%)", fontsize=8)
frame(ax)
fig.tight_layout()
fig.savefig("fig_10_7.png", dpi=300)
plt.show()

In [ ]:
def with_floor(unit, lam):
    """층을 unit으로 나눠서 넣었을 때의 층 계수(1층 기준으로 환산)와 2024년 오차율. unit이 10이면 12층이 1.2가 된다."""
    T, T24 = S.tolil(copy=True), S24.tolil(copy=True)
    T[:, 1], T24[:, 1] = S[:, 1] / unit, S24[:, 1] / unit
    m = Ridge(alpha=lam, solver="sparse_cg", tol=1e-8, max_iter=20000).fit(T.tocsr(), y23)
    return round(float(m.coef_[1] / unit), 5), off(y24, m.predict(T24.tocsr()))

for lam in [0.01, 100]:
    print(lam, with_floor(1, lam), with_floor(10, lam), with_floor(100, lam))

In [ ]:
DB, CB = sorted(both["동이름"].unique()), sorted(both["단지"].unique())
SB, S25 = design(both, DB, CB), design(a25, DB, CB)
yb, y25 = np.log(both["억"].to_numpy()), np.log(a25["억"].to_numpy())
last = Ridge(alpha=0.01, solver="sparse_cg", tol=1e-8, max_iter=20000).fit(SB, yb)
print(SB.shape, off(yb, last.predict(SB)), off(y25, last.predict(S25)))

new = ~a25["단지"].isin(CB).to_numpy()                       # 2024년까지 거래가 없었던 단지의 거래
print(int(new.sum()), off(y25[new], last.predict(S25)[new]), off(y25[~new], last.predict(S25)[~new]))
only_dong = Ridge(alpha=0.01).fit(SB[:, : 7 + len(DB)], yb)
print(off(y25, only_dong.predict(S25[:, : 7 + len(DB)])), off(y25[new], only_dong.predict(S25[:, : 7 + len(DB)])[new]))

## 10.3 규제가 필요한 자료: 잡음과 신호의 비

In [ ]:
def first_line(path):
    with open(path, encoding="cp949", errors="ignore") as f:
        return f.readline()

name = [f for f in glob.glob("*.csv") if first_line(f).startswith("시군명,사업장명")][0]
raw = pd.read_csv(name, encoding="cp949", dtype=str)

raw["시군"] = raw["시군명"].str.strip()
raw["구분"] = np.where(raw["시군명"].str.endswith(" "), "공백 있음", "공백 없음")
shops = raw.sort_values(["구분", "영업상태명"], ascending=[True, False]).drop_duplicates(["시군", "사업장명", "인허가일자"]).copy()

shops["개업"] = pd.to_datetime(shops["인허가일자"], format="%Y%m%d", errors="coerce")
shops["폐업"] = pd.to_datetime(shops["폐업일자"], format="%Y%m%d", errors="coerce")
shops = shops.dropna(subset=["개업"]).reset_index(drop=True)
END = max(shops["개업"].max(), shops["폐업"].max())
shops["끝"] = shops["폐업"].fillna(END + pd.Timedelta(days=1))
shops["위도"] = pd.to_numeric(shops["WGS84위도"], errors="coerce")
shops["경도"] = pd.to_numeric(shops["WGS84경도"], errors="coerce")
shops["지번"] = shops["소재지지번주소"].str.replace(r"\s+", " ", regex=True).str.extract(r"^(.*?\d+(?:-\d+)?)")[0]
print(len(raw), len(shops), END.date())

In [ ]:
def snapshot(day):
    """day에 영업 중이던 가게마다 한 줄. 특성은 그날 알 수 있는 값으로만 만들고, 그 뒤 3년 안에 폐업했는지를 레이블로 붙인다."""
    day = pd.Timestamp(day)
    began, ended = shops["개업"].to_numpy(), shops["끝"].to_numpy()
    open_now = (began < day.to_datetime64()) & (ended > day.to_datetime64())
    gone = shops["폐업"].notna().to_numpy() & (ended < day.to_datetime64())
    lately = gone & (ended >= (day - pd.DateOffset(years=3)).to_datetime64())

    a = shops[open_now].copy()
    a["기준일"] = day
    a["업력"] = (day - a["개업"]).dt.days / 365.25
    a["지번폐업"] = a["지번"].map(shops[gone].groupby("지번").size()).fillna(0)      # 같은 지번에서 먼저 폐업한 중국집 수

    def within(mask, metres):
        """a의 가게마다, mask에 해당하는 가게 가운데 metres 안에 있는 것의 수."""
        lat, lon = shops.loc[mask, "위도"].to_numpy(), shops.loc[mask, "경도"].to_numpy()
        dy = (a["위도"].to_numpy()[:, None] - lat[None, :]) * 111195
        dx = (a["경도"].to_numpy()[:, None] - lon[None, :]) * 111195 * np.cos(np.radians(a["위도"].to_numpy()))[:, None]
        return np.where(a["위도"].isna(), np.nan, (dx ** 2 + dy ** 2 < metres ** 2).sum(axis=1))

    a["주변"] = within(open_now, 500) - 1            # 반경 500m 안에서 영업 중인 다른 중국집
    a["주변폐업"] = within(lately, 1000)              # 반경 1km 안에서 지난 3년 사이 폐업한 중국집
    a["폐업함"] = (a["폐업"].notna() & (a["폐업"] <= day + pd.DateOffset(years=3))).astype(int)
    return a

train = pd.concat([snapshot(d) for d in ["2011-04-03", "2014-04-03", "2017-04-03", "2020-04-03"]], ignore_index=True)
test = snapshot("2023-04-03").reset_index(drop=True)
y = test["폐업함"].to_numpy()

print(train.groupby("기준일")["폐업함"].agg(["size", "mean"]).round(3).to_string())
print(len(test), int(y.sum()), round(float(y.mean()), 3))

In [ ]:
SIGUN = sorted(shops["시군"].unique())
YEARS = [0, 1, 2, 3, 5, 10, 20]

def onehot(s, cats):
    return (np.asarray(s)[:, None] == np.asarray(cats)[None, :]).astype(float)

def columns(df):
    """업력 구간 가변수 7개, 시군 가변수 30개, 같은 지번의 폐업 이력, 주변의 영업 중·최근 폐업 가게 수(로그), 좌표 없음 표시."""
    return np.column_stack([
        onehot(np.digitize(df["업력"], YEARS) - 1, range(len(YEARS))),
        onehot(df["시군"], SIGUN)[:, 1:],
        np.minimum(df["지번폐업"], 3),
        np.log1p(df["주변"].fillna(0)), np.log1p(df["주변폐업"].fillna(0)), df["주변"].isna().astype(float),
    ])

def auc(actual, s):
    rank = pd.Series(s).rank().to_numpy()
    n1 = actual.sum()
    return float((rank[actual == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * (len(actual) - n1)))

X, Xt, ytr = columns(train), columns(test), train["폐업함"].to_numpy()
score = Xt @ np.linalg.lstsq(X, ytr, rcond=None)[0]
print(X.shape, round(auc(y, score), 3))

In [ ]:
def town(df):
    """지번 주소에서 시군(구)과 읍면동까지만 잘라 낸다."""
    addr = df["소재지지번주소"].fillna("").str.replace(r"\s+", " ", regex=True)
    return addr.str.extract(r"^경기도 (\S+(?: \S+구)? \S+?[읍면동가리])(?:\s|$)")[0].fillna("?")

year = train["기준일"].dt.year.to_numpy()
fit_on, tune_on = year < 2020, year == 2020                # 계수를 추정하는 줄, 규제 세기를 고르는 줄
TOWNS = sorted(town(train)[fit_on].unique())
W, Wt = np.column_stack([X, onehot(town(train), TOWNS)]), np.column_stack([Xt, onehot(town(test), TOWNS)])
per_town = town(train)[fit_on].value_counts()
print(W.shape, int(fit_on.sum()), int(tune_on.sum()), len(TOWNS), int(per_town.median()), int((per_town <= 5).sum()))

def brier(actual, prob):
    return float(((prob - actual) ** 2).mean())

def report(name, M, Mt, lam):
    coef, base = ridge(M[fit_on], ytr[fit_on], lam)
    said, said_t = M[tune_on] @ coef + base, Mt @ coef + base
    print(f"{name}: 2020년 AUC {auc(ytr[tune_on], said):.3f} 브라이어 {brier(ytr[tune_on], said):.4f}")
    return said_t

rate = ytr[fit_on].mean()
print(f"기준율로만 예측: 2020년 브라이어 {brier(ytr[tune_on], rate):.4f}")
t41 = report("특성 41개", X, Xt, 1e-6)
t679 = report("읍면동을 더한 특성 679개", W, Wt, 1e-6)

In [ ]:
LAMS2 = [1, 3, 10, 30, 100, 300, 1000, 3000]
centered = W[fit_on] - W[fit_on].mean(axis=0)
d2 = np.linalg.svd(centered, compute_uv=False) ** 2        # 특잇값의 제곱
rows = []
for lam in LAMS2:
    coef, base = ridge(W[fit_on], ytr[fit_on], lam)
    said = W[tune_on] @ coef + base
    rows.append((lam, round(auc(ytr[tune_on], said), 3), round(brier(ytr[tune_on], said), 4), round(float(np.abs(coef[41:]).mean()), 4),
                 round(float((d2 / (d2 + lam)).sum()), 1)))
shop = pd.DataFrame(rows, columns=["λ", "2020년 AUC", "2020년 브라이어", "읍면동 계수의 크기", "유효 자유도"]).set_index("λ")
print(shop.to_string())
best_lam = int(shop["2020년 브라이어"].idxmin())
print(best_lam, int((d2 > 1e-8).sum()))

In [ ]:
# 그림 10-8
fig, ax = plt.subplots(figsize=(7, 3.9))
b41 = brier(ytr[tune_on], X[tune_on] @ ridge(X[fit_on], ytr[fit_on], 1e-6)[0] + ridge(X[fit_on], ytr[fit_on], 1e-6)[1])
b679 = brier(ytr[tune_on], W[tune_on] @ ridge(W[fit_on], ytr[fit_on], 1e-6)[0] + ridge(W[fit_on], ytr[fit_on], 1e-6)[1])
b0 = brier(ytr[tune_on], rate)
ax.plot(shop.index, shop["2020년 브라이어"], color=NAVY, lw=1.8, marker="o", ms=5)
ax.axhline(b679, color=BRICK, lw=1.1, ls=(0, (4, 2)))
ax.axhline(b0, color=GRAY, lw=1.1, ls=(0, (4, 2)))
ax.axhline(b41, color=TEAL, lw=1.1, ls=(0, (4, 2)))
ax.text(3000, b679 + 0.00015, f"특성 679개, 규제 없음 {b679:.4f}", fontsize=8, color=BRICK, ha="right", va="bottom")
ax.text(3000, b0 + 0.00015, f"기준율로만 예측 {b0:.4f}", fontsize=8, color="#5b6570", ha="right", va="bottom")
ax.text(1, b41 - 0.00022, f"특성 41개 {b41:.4f}", fontsize=8, color=TEAL, ha="left", va="top")
ax.annotate(f"λ = {best_lam}에서 {shop['2020년 브라이어'].min():.4f}", (best_lam, shop["2020년 브라이어"].min()), (25, 0.1455), fontsize=8, color=NAVY, arrowprops={"arrowstyle": "->", "color": NAVY, "lw": 0.8})
ax.text(1.6, 0.1522, "특성 679개 + 릿지", fontsize=8, color=NAVY)
ax.set_xscale("log"); ax.set_ylim(0.145, 0.1548)
plain_ticks(ax.xaxis)
ax.set_xlabel("규제 세기 λ(로그 축)", fontsize=8)
ax.set_ylabel("2020년 검증 자료의 브라이어 점수", fontsize=8)
frame(ax)
fig.tight_layout()
fig.savefig("fig_10_8.png", dpi=300)
plt.show()

In [ ]:
def noise_and_signal(resid, group):
    """범주 안에서의 분산(잡음), 범주 평균들 사이의 분산(신호), 그리고 잡음을 신호로 나눈 값."""
    g = pd.DataFrame({"g": np.asarray(group), "r": resid}).groupby("g")["r"].agg(["mean", "size", "var"])
    noise = float((g["var"].fillna(0) * (g["size"] - 1)).sum() / (g["size"].sum() - len(g)))
    spread = float((g["size"] * (g["mean"] - resid.mean()) ** 2).sum())
    weight = g["size"].sum() - (g["size"] ** 2).sum() / g["size"].sum()
    signal = (spread - (len(g) - 1) * noise) / weight          # 평균이 표본 수 때문에 달라지는 부분을 뺀다
    return round(noise, 4), round(float(signal), 4), round(float(noise / signal), 2)

apt_ratio = noise_and_signal(left23, a23["단지"])
coef, base = ridge(X[fit_on], ytr[fit_on], 1e-6)
shop_ratio = noise_and_signal(ytr[fit_on] - (X[fit_on] @ coef + base), town(train)[fit_on])
print("아파트 단지", apt_ratio, round(1 / (1 + apt_ratio[2]), 2))
print("중국집 읍면동", shop_ratio, round(11 / (11 + shop_ratio[2]), 2))

In [ ]:
# 그림 10-9
fig, ax = plt.subplots(figsize=(7, 3.9))
ns = np.arange(1, 1001)
for ratio, col in [(apt_ratio[2], NAVY), (shop_ratio[2], BRICK)]:
    ax.plot(ns, ns / (ns + ratio), color=col, lw=1.8)
ax.scatter([3], [3 / (3 + apt_ratio[2])], s=46, color=NAVY, zorder=5)
ax.scatter([11], [11 / (11 + shop_ratio[2])], s=46, color=BRICK, zorder=5)
ax.annotate(f"단지의 거래 수 중앙값 3건: {3 / (3 + apt_ratio[2]):.0%}", (3, 3 / (3 + apt_ratio[2])), (5.5, 0.74), fontsize=8, color=NAVY, arrowprops={"arrowstyle": "->", "color": NAVY, "lw": 0.8})
ax.annotate(f"읍면동의 줄 수 중앙값 11줄: {11 / (11 + shop_ratio[2]):.0%}", (11, 11 / (11 + shop_ratio[2])), (1.3, 0.2), fontsize=8, color=BRICK, arrowprops={"arrowstyle": "->", "color": BRICK, "lw": 0.8})
ax.text(40, 0.91, f"아파트 단지: 잡음/신호 = {apt_ratio[2]}", fontsize=8, color=NAVY)
ax.text(160, 0.42, f"중국집 읍면동:\n잡음/신호 = {shop_ratio[2]:.0f}", fontsize=8, color=BRICK, linespacing=1.5)
ax.set_xscale("log"); ax.set_ylim(0, 1.04); ax.set_xlim(0.9, 1100)
plain_ticks(ax.xaxis)
ax.set_xlabel("범주 하나에 속한 줄 수 n(로그 축)", fontsize=8)
ax.set_ylabel("범주 평균을 반영하는 비율 n / (n + λ)", fontsize=8)
frame(ax)
fig.tight_layout()
fig.savefig("fig_10_9.png", dpi=300)
plt.show()

## 10.4 라쏘: 계수를 0으로 만드는 규제

In [ ]:
# 그림 10-11
fig, ax = plt.subplots(figsize=(7, 4.2))
z = np.linspace(-3, 3, 601)
ax.plot(z, z, color=GRAY, lw=1.2, ls=(0, (4, 2)), label="규제 없음")
ax.plot(z, z / 2, color=NAVY, lw=1.9, label="릿지(λ = 1): 같은 비율로 줄인다")
ax.plot(z, np.sign(z) * np.maximum(np.abs(z) - 1, 0), color=BRICK, lw=1.9, label="라쏘(α = 1): 같은 크기만큼 줄인다")
ax.axhline(0, color="#c9d1da", lw=0.8, zorder=0); ax.axvline(0, color="#c9d1da", lw=0.8, zorder=0)
ax.legend(frameon=False, fontsize=8, loc="lower right")
ax.annotate("이 구간에서는 계수가 정확히 0", (-0.5, 0.04), (-2.9, 0.9), fontsize=8, color=BRICK, arrowprops={"arrowstyle": "->", "color": BRICK, "lw": 0.8})
ax.plot([-1, 1], [0, 0], color=BRICK, lw=4, solid_capstyle="butt", zorder=4)
ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
ax.set_xlabel("규제가 없을 때의 계수", fontsize=8)
ax.set_ylabel("규제를 건 뒤의 계수", fontsize=8)
frame(ax, grid="both")
fig.tight_layout()
fig.savefig("fig_10_11.png", dpi=300)
plt.show()

In [ ]:
def lasso(M, target, alpha, laps=300):
    """제곱 오차 평균의 절반 + alpha × (계수 절댓값의 합)을 줄인다. 계수를 하나씩 차례로 고친다(좌표 하강법)."""
    mid, top = M.mean(axis=0), target.mean()
    C_, left = M - mid, target - top
    m = len(target)
    size = (C_ ** 2).sum(axis=0) / m
    coef = np.zeros(M.shape[1])
    for lap in range(laps):
        moved = 0.0
        for j in np.where(size > 0)[0]:
            pull = C_[:, j] @ left / m + size[j] * coef[j]              # 다른 계수를 고정했을 때 이 특성과 잔차의 공분산
            new = np.sign(pull) * max(abs(pull) - alpha, 0) / size[j]   # 크기가 alpha보다 작으면 0이 된다
            if new != coef[j]:
                left -= C_[:, j] * (new - coef[j])
                moved = max(moved, abs(new - coef[j]))
                coef[j] = new
        if moved < 1e-7:
            break
    return coef, top - mid @ coef, lap + 1

from sklearn.linear_model import Lasso

coef, base, laps = lasso(W[fit_on], ytr[fit_on], 0.0005)
theirs = Lasso(alpha=0.0005, max_iter=5000, tol=1e-8).fit(W[fit_on], ytr[fit_on])
print(laps, int((coef != 0).sum()), int((coef[:41] != 0).sum()), int((coef[41:] != 0).sum()), f"{np.abs(coef - theirs.coef_).max():.1e}")

In [ ]:
ALPHAS = [0.003, 0.002, 0.001, 0.0005, 0.0003, 0.0001, 0.00003]
rows, nets = [], {}
for alpha in ALPHAS:
    nets[alpha] = Lasso(alpha=alpha, max_iter=20000, tol=1e-7).fit(W[fit_on], ytr[fit_on])
    said = nets[alpha].predict(W[tune_on])
    rows.append((alpha, int((nets[alpha].coef_ != 0).sum()), int((nets[alpha].coef_[41:] != 0).sum()),
                 round(auc(ytr[tune_on], said), 3), round(brier(ytr[tune_on], said), 4)))
trail = pd.DataFrame(rows, columns=["α", "남은 특성", "그중 읍면동", "2020년 AUC", "2020년 브라이어"]).set_index("α")
print(trail.to_string())

NAMES = [f"업력 {a}년~" for a in YEARS] + SIGUN[1:] + ["지번폐업", "주변", "주변폐업", "좌표없음"] + TOWNS
kept = pd.Series(nets[0.001].coef_, index=NAMES)
print(kept[kept != 0].round(4).to_string())

In [ ]:
# 그림 10-12
fig, axes = plt.subplots(2, 1, figsize=(7, 5.6), sharex=True)
xs_ = np.arange(len(ALPHAS))
axes[0].bar(xs_, trail["남은 특성"], width=0.62, color=to_rgba(NAVY, 0.22), edgecolor=NAVY, lw=1)
axes[0].bar(xs_, trail["그중 읍면동"], width=0.62, color=to_rgba(BRICK, 0.55), edgecolor=BRICK, lw=1)
for i, (a, b_) in enumerate(zip(trail["남은 특성"], trail["그중 읍면동"])):
    axes[0].text(i, a + 14, f"{a}개", ha="center", fontsize=8, color=NAVY)
axes[0].text(3.9, 420, "전체 막대: 계수가 0이 아닌 특성의 수", fontsize=8, color=NAVY, ha="right")
axes[0].text(3.9, 365, "벽돌색: 그 가운데 읍면동 가변수", fontsize=8, color=BRICK, ha="right")
axes[0].set_ylim(0, 600); axes[0].set_ylabel("남은 특성의 수", fontsize=8)
axes[0].tick_params(labelbottom=True)
frame(axes[0], "규제 세기 α에 따라 남는 특성의 수")
axes[1].plot(xs_, trail["2020년 브라이어"], color=NAVY, lw=1.8, marker="o", ms=5)
axes[1].axhline(b0, color=GRAY, lw=1.1, ls=(0, (4, 2)))
axes[1].text(0, b0 + 0.00012, f"기준율로만 예측 {b0:.4f}", fontsize=8, color="#5b6570", va="bottom", ha="left")
for i, b_ in enumerate(trail["2020년 브라이어"]):
    if i < len(ALPHAS) - 1:
        axes[1].text(i, b_ - 0.00042, f"{b_:.4f}", ha="center", fontsize=8, color=NAVY)
    else:
        axes[1].text(i - 0.15, b_, f"{b_:.4f}", ha="right", va="center", fontsize=8, color=NAVY)
axes[1].set_ylim(0.1458, 0.1509); axes[1].set_ylabel("2020년 브라이어 점수", fontsize=8)
axes[1].set_xticks(xs_, [f"{a:.5f}".rstrip("0") for a in ALPHAS])
axes[1].set_xlabel("규제 세기 α (오른쪽으로 갈수록 규제가 약하다)", fontsize=8)
frame(axes[1], "2020년 검증 자료의 브라이어 점수")
fig.tight_layout(h_pad=1.4)
fig.savefig("fig_10_12.png", dpi=300)
plt.show()

In [ ]:
ids = (train["시군"] + "|" + train["사업장명"] + "|" + train["인허가일자"]).to_numpy()[fit_on]
shop_ids, which = np.unique(ids, return_inverse=True)
rows_of = [np.where(which == g)[0] for g in range(len(shop_ids))]
We, ye = W[fit_on], ytr[fit_on]

rng = np.random.default_rng(0)
chosen = np.zeros(W.shape[1])
sizes = []
for _ in range(40):
    rows = np.concatenate([rows_of[g] for g in rng.integers(0, len(shop_ids), len(shop_ids))])     # 8장처럼 가게 단위로 복원 추출한다
    again = Lasso(alpha=0.0005, max_iter=20000, tol=1e-7).fit(We[rows], ye[rows]).coef_ != 0
    chosen += again
    sizes.append(int(again.sum()))
print(len(shop_ids), min(sizes), int(np.median(sizes)), max(sizes))
print(int((chosen == 40).sum()), int((chosen >= 36).sum()), int((chosen >= 20).sum()), int((chosen >= 1).sum()))
first = nets[0.0005].coef_ != 0
often = pd.Series(chosen[first], index=np.array(NAMES)[first]).astype(int).sort_values()
print(int(first.sum()), int((often < 20).sum()), int((often >= 36).sum()), often.head(10).to_dict())

In [ ]:
# 그림 10-13
fig, ax = plt.subplots(figsize=(7, 6.6))
order = often.sort_values(ascending=True)
is_town = np.array([nm in TOWNS for nm in order.index])
cols = [to_rgba(BRICK, 0.55) if t else to_rgba(NAVY, 0.22) for t in is_town]
edges = [BRICK if t else NAVY for t in is_town]
ax.barh(range(len(order)), order.to_numpy(), color=cols, edgecolor=edges, lw=0.9, height=0.68)
for i, c in enumerate(order.to_numpy()):
    ax.text(c + 0.5, i, f"{c}", va="center", fontsize=7.5, color="#44505c", bbox=BOX, zorder=4)
ax.axvline(20, color=GRAY, lw=1, ls=(0, (4, 2)))
ax.text(20, len(order) - 0.25, "40번 중 20번", fontsize=8, color="#5b6570", ha="center", va="bottom", bbox=BOX)
ax.set_yticks(range(len(order)), order.index, fontsize=7.5)
ax.set_xlim(0, 44); ax.set_ylim(-0.7, len(order) + 0.5)
ax.set_xlabel("부트스트랩 40번 가운데 계수가 0이 아니었던 횟수", fontsize=8)
ax.text(43.5, 3.2, "남색: 원래 특성 41개에 속한 것\n벽돌색: 읍면동 가변수", fontsize=8, color="#44505c", ha="right", linespacing=1.6, bbox=BOX)
frame(ax, grid="x")
ax.tick_params(axis="y", labelsize=7.5)
fig.tight_layout()
fig.savefig("fig_10_13.png", dpi=300)
plt.show()

In [ ]:
def exam(name, said):
    print(f"{name}: 2023년 AUC {auc(y, said):.3f} 브라이어 {brier(y, said):.4f}")

exam("특성 41개", t41)
exam("특성 679개, 규제 없음", t679)
coef, base = ridge(W[fit_on], ytr[fit_on], best_lam)
t_ridge = Wt @ coef + base
exam(f"특성 679개, 릿지 λ={best_lam}", t_ridge)
t_lasso = nets[0.0005].predict(Wt)
exam("특성 679개, 라쏘 α=0.0005", t_lasso)
print(f"기준율로만 예측: 브라이어 {brier(y, rate):.4f}")

In [ ]:
rng = np.random.default_rng(0)
gaps = []
for _ in range(2000):
    pick = rng.integers(0, len(y), len(y))                 # 8.2절의 짝지은 부트스트랩: 같은 가게를 뽑아 두 모델을 함께 평가한다
    a41 = auc(y[pick], t41[pick])
    gaps.append((a41 - auc(y[pick], t679[pick]), a41 - auc(y[pick], t_ridge[pick]), a41 - auc(y[pick], t_lasso[pick])))
gaps = np.array(gaps)
for name, col in zip(["규제 없음", "릿지", "라쏘"], gaps.T):
    print(f"특성 41개 - 특성 679개({name}): AUC 차이 {col.mean():+.3f} 95% 구간 [{np.quantile(col, 0.025) + 0:+.3f}, {np.quantile(col, 0.975) + 0:+.3f}]")